In [1]:
from ts_toolkit.pipeline.runner import TsPipeline
from ts_toolkit.pipeline.utils import load_config
import logging
from tqdm.notebook import tqdm
import pandas as pd
from dotenv import load_dotenv
import os
import numpy as np
from gluonts.model.forecast import QuantileForecast
import utilsforecast.processing as ufp
from gluonts.model import Forecast
from multivar_timeseries.evaluation.metrics import MAR, MBR

# auto reload imports in jupyter
%load_ext autoreload
%autoreload 2

In [2]:
load_dotenv()
print(GIFT_EVAL_PATH := os.getenv("GIFT_EVAL"))
print(REPO_ROOT := os.getenv("REPO_ROOT"))

QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

/home/roc/01_projects/11_time-series/timeseries-research/data/gift_eval
/home/roc/01_projects/11_time-series/timeseries-research


In [10]:
logging.basicConfig(level=logging.INFO)

# --- Load configuration ---
# config_path = "./configs/experiments/simpletime.yaml"
# config_path = "../configs/experiments/simpletime_sample.yaml"
config_path = "configs/experiments/personal/gifteval_roc_test.yaml"
# config_path = "./configs/experiments/gifteval.yaml"

config = load_config(config_path)

# --- Set up the evaluation pipeline ---
pipeline = TsPipeline(config)
pipeline.run(skip_existing=False)

pipeline.get_results()


INFO:root:   - Results will be saved to: /home/roc/01_projects/11_time-series/timeseries-research/results/roc_test_20251008-182336/ToTo-GiftEval-sample
INFO:root:   - Datasets will be loaded from: /home/roc/01_projects/11_time-series/timeseries-research/data/gift_eval

INFO:root:
--- Starting Evaluation ---


Loading base config from <REPO_ROOT>/configs/experiments/personal/gifteval_roc_test.yaml...


Evaluating Datasets:   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 42/42 [00:00<00:00, 3712.56it/s]
42it [00:00, 1195.91it/s]
INFO:ts_toolkit.pipeline.eval:Results for bizitobs_l2c/H have been written to /home/roc/01_projects/11_time-series/timeseries-research/results/roc_test_20251008-182336/ToTo-GiftEval-sample/all_results.csv
INFO:root:
--- ✅ All evaluations complete. ---


,dataset,variate,num_variates,model,eval_metrics/MAR[0.5][512],eval_metrics/MBR[0.5][512],eval_metrics/MSE[mean],eval_metrics/MSE[0.5],eval_metrics/MAE[0.5],eval_metrics/MASE[0.5],eval_metrics/MAPE[0.5],eval_metrics/sMAPE[0.5],eval_metrics/MSIS,eval_metrics/RMSE[mean],eval_metrics/NRMSE[mean],eval_metrics/ND[0.5],eval_metrics/mean_weighted_sum_quantile_loss
0,bizitobs_l2c/H/short,UNI,7,ToTo-GiftEval-sample,1.376592,0.728671,131.543331,131.543331,7.071794,0.69759,0.635067,0.854105,4.70934,11.469234,0.618221,0.381188,0.287757


In [23]:
config.experiment_name

'roc_test_20251008-170210'

In [12]:
import pandas as pd
df_results = pd.read_csv(f"{REPO_ROOT}/results/{config.experiment_name}/{config.forecaster.alias}/all_results.csv")
df_results

,dataset,variate,num_variates,model,eval_metrics/MSE[mean],eval_metrics/MSE[0.5],eval_metrics/MAE[0.5],eval_metrics/MASE[0.5],eval_metrics/MAPE[0.5],eval_metrics/sMAPE[0.5],eval_metrics/MSIS,eval_metrics/RMSE[mean],eval_metrics/NRMSE[mean],eval_metrics/ND[0.5],eval_metrics/mean_weighted_sum_quantile_loss
0,bizitobs_l2c/H,UNI,7,ToTo-GiftEval-sample,69.697307,69.697307,5.066153,0.504449,0.438327,0.685013,3.385132,8.348491,0.450005,0.273079,0.214369


# Investigate

## Get Forecasts

In [4]:
import pandas as pd
import logging
from tqdm.notebook import tqdm
from ts_toolkit.models.ensembles.median import MedianEnsemble
from ts_toolkit.models import MULTIVARIATE_MODEL_LIST
from ts_toolkit.pipeline.gluonts_predictor import GluonTSPredictor
from ts_toolkit.pipeline.eval import Evaluator
from ts_toolkit.models import MODEL_REGISTRY, get_model
import os


In [5]:
dataset_cfg = pipeline.config.datasets[0]
to_univariate = pipeline.config.evaluation.to_univariate


In [6]:
evaluator = Evaluator(
    dataset_name=dataset_cfg.get("name"),
    term=dataset_cfg.get("term", "short"),
    to_univariate=to_univariate,
    # Create a subdirectory for each model's results
    output_path=f"{pipeline.output_path}/{pipeline.forecaster.alias}",
    storage_path=pipeline.storage_path,
    config=pipeline.config
)


In [7]:
dataset = evaluator.dataset.test_data.input
dataset

InputDataset(test_data=TestData(dataset=<gluonts.transform._base.TransformedDataset object at 0x716b8048ab10>, splitter=OffsetSplitter(offset=-288), prediction_length=48, windows=6, distance=48, max_history=None))

In [8]:
h = pipeline.predictor.h or dataset.test_data.prediction_length
freq = pipeline.predictor.freq
print(h, freq, pipeline.predictor.batch_size)

48 None 1024


In [9]:
evaluator.forecasts = pipeline.predictor.predict(dataset)

100%|██████████| 42/42 [00:00<00:00, 2682.35it/s]


In [10]:
evaluator.forecasts.__len__(), [len(x.mean) for x in evaluator.forecasts][0]

(42, 48)

In [11]:
batch = []
for _, entry in enumerate(dataset):
    batch.append(entry)

# So batch is w * n_var * T
len(batch), [wind['target'].shape for wind in batch]

(42,
 [(2376,),
  (2424,),
  (2472,),
  (2520,),
  (2568,),
  (2616,),
  (2376,),
  (2424,),
  (2472,),
  (2520,),
  (2568,),
  (2616,),
  (2376,),
  (2424,),
  (2472,),
  (2520,),
  (2568,),
  (2616,),
  (2376,),
  (2424,),
  (2472,),
  (2520,),
  (2568,),
  (2616,),
  (2376,),
  (2424,),
  (2472,),
  (2520,),
  (2568,),
  (2616,),
  (2376,),
  (2424,),
  (2472,),
  (2520,),
  (2568,),
  (2616,),
  (2376,),
  (2424,),
  (2472,),
  (2520,),
  (2568,),
  (2616,)])

In [12]:
# Extract all values for each variate across all windows
variates = batch[0]['target'].shape[0]  # number of variates
windows = len(batch)

# var_i: all values of variate i from all windows concatenated
var_dict = {}
for i in range(variates):
    var_dict[f'var_{i+1}'] = [batch[w]['target'][i].flatten() for w in range(windows)]
    var_dict[f'var_{i+1}'] = np.concatenate(var_dict[f'var_{i+1}'])

# var_i_j: all values of variate i from window j (j=1-based)
for i in range(variates):
    for j in range(windows):
        var_dict[f'var_{i+1}_{j+1}'] = batch[j]['target'][i].flatten()

# Example: var_1 contains all values of variate 1 from all windows
#          var_1_1 contains all values of variate 1 from window 1

# Check if the first 2376 values of all windows are the same for each variate
same_across_windows = {}
for i in range(variates):
    key = f'var_{i+1}_1'
    reference = var_dict[key][:2376]
    all_equal = True
    for j in range(2, windows+1):
        compare_key = f'var_{i+1}_{j}'
        if not np.array_equal(reference, var_dict[compare_key][:2376]):
            all_equal = False
            break
    same_across_windows[f'var_{i+1}'] = all_equal

same_across_windows

{'var_1': True,
 'var_2': True,
 'var_3': True,
 'var_4': True,
 'var_5': True,
 'var_6': True,
 'var_7': True,
 'var_8': True,
 'var_9': True,
 'var_10': True,
 'var_11': True,
 'var_12': True,
 'var_13': True,
 'var_14': True,
 'var_15': True,
 'var_16': True,
 'var_17': True,
 'var_18': True,
 'var_19': True,
 'var_20': True,
 'var_21': True,
 'var_22': True,
 'var_23': True,
 'var_24': True,
 'var_25': False,
 'var_26': False,
 'var_27': False,
 'var_28': False,
 'var_29': False,
 'var_30': False,
 'var_31': False,
 'var_32': False,
 'var_33': False,
 'var_34': False,
 'var_35': False,
 'var_36': False,
 'var_37': False,
 'var_38': False,
 'var_39': False,
 'var_40': False,
 'var_41': False,
 'var_42': False,
 'var_43': False,
 'var_44': False,
 'var_45': False,
 'var_46': False,
 'var_47': False,
 'var_48': False,
 'var_49': False,
 'var_50': False,
 'var_51': False,
 'var_52': False,
 'var_53': False,
 'var_54': False,
 'var_55': False,
 'var_56': False,
 'var_57': False,
 'var_5

### Findings:
> 1. evaluator.dataset.test_data.input, which converts to the batch input, is in the shape of (n_windows, n_var, n_time_step), Each window and var contains all values of the previous window + 48 new time steps, which is the `h`
> 2. df is just all n_windows concatenated together vertically, each window with the same starting ds. So the number of rows of df = sum of all time steps in the (test) dataset. Multivariates are converted into y_i columns
> 3. predict_df takes df to run MedianEnsemble.forecast(), and returns fcst_df in the shape of (n_windows x n_var x h,  2 (unique_id, ds) + 1 (median) + 9 (quantiles)) or (6 x 7 x 48, 12)
> 4. Last bit of predict_df wrangles fcst_df to the shape of (n_windows x n_var, h) or (6 * 7, 48)

In [13]:
df = pipeline.predictor._gluonts_dataset_to_df(batch)
df.shape

(21504, 3)

In [14]:
df

,unique_id,ds,y
0,L2C_dim0-2021-04-26 16:00,2021-01-17 16:00:00,0.000000
1,L2C_dim0-2021-04-26 16:00,2021-01-17 17:00:00,0.000000
2,L2C_dim0-2021-04-26 16:00,2021-01-17 18:00:00,0.000000
3,L2C_dim0-2021-04-26 16:00,2021-01-17 19:00:00,0.000000
4,L2C_dim0-2021-04-26 16:00,2021-01-17 20:00:00,0.000000
...,...,...,...
21499,L2C_dim6-2021-05-16 16:00,2021-02-17 19:00:00,50.000000
21500,L2C_dim6-2021-05-16 16:00,2021-02-17 20:00:00,49.333332
21501,L2C_dim6-2021-05-16 16:00,2021-02-17 21:00:00,45.000000
21502,L2C_dim6-2021-05-16 16:00,2021-02-17 22:00:00,44.333332


In [15]:
metadata = pipeline.predictor.multivar_metadata if len(pipeline.predictor.target_cols) > 1 else pipeline.predictor.univar_metadata

In [16]:
fcst_df = pipeline.predictor.forecaster.forecast(
            df=df,
            h=h,
            freq=freq,
            level=pipeline.predictor.level,
            quantiles=pipeline.predictor.quantiles,
        )

100%|██████████| 1/1 [00:02<00:00,  2.30s/it]


In [17]:
fcst_df

,unique_id,ds,ToTo-GiftEval-sample,ToTo-GiftEval-sample-q-10,ToTo-GiftEval-sample-q-20,ToTo-GiftEval-sample-q-30,ToTo-GiftEval-sample-q-40,ToTo-GiftEval-sample-q-50,ToTo-GiftEval-sample-q-60,ToTo-GiftEval-sample-q-70,ToTo-GiftEval-sample-q-80,ToTo-GiftEval-sample-q-90
0,L2C_dim0-2021-04-26 16:00,2021-02-08 00:00:00,-0.011784,-0.145235,-0.103537,-0.072051,-0.038383,-0.011784,0.023573,0.049729,0.081593,0.128888
1,L2C_dim0-2021-04-26 16:00,2021-02-08 01:00:00,-0.003800,-0.173691,-0.130952,-0.063952,-0.036861,-0.003800,0.017765,0.051220,0.095370,0.256618
2,L2C_dim0-2021-04-26 16:00,2021-02-08 02:00:00,0.018100,-0.109281,-0.068213,-0.035481,-0.011383,0.018100,0.047198,0.097675,0.161755,0.433783
3,L2C_dim0-2021-04-26 16:00,2021-02-08 03:00:00,0.025383,-0.196389,-0.094451,-0.043426,-0.011423,0.025383,0.062298,0.107363,0.209812,0.936370
4,L2C_dim0-2021-04-26 16:00,2021-02-08 04:00:00,0.041791,-0.199245,-0.112254,-0.041033,-0.003540,0.041791,0.094614,0.184517,0.427060,1.400816
...,...,...,...,...,...,...,...,...,...,...,...,...
2011,L2C_dim6-2021-05-16 16:00,2021-02-19 19:00:00,33.645485,17.759678,24.500380,27.473164,31.113836,33.645485,36.469494,39.539562,44.023434,49.831783
2012,L2C_dim6-2021-05-16 16:00,2021-02-19 20:00:00,34.171158,16.377939,24.423496,28.485098,31.527706,34.171158,37.055298,40.684578,45.593807,53.116810
2013,L2C_dim6-2021-05-16 16:00,2021-02-19 21:00:00,32.619537,19.260370,25.395956,28.226646,30.806803,32.619537,34.875942,39.798557,43.566513,49.607658
2014,L2C_dim6-2021-05-16 16:00,2021-02-19 22:00:00,33.682571,20.359383,26.076817,29.754988,31.679110,33.682571,37.213833,38.580345,42.679256,48.586823


In [18]:
fcst_df.shape

(2016, 12)

In [19]:
2016/48

42.0

In [20]:
fcst_df = fcst_df.set_index("unique_id")
fcsts: list[Forecast] = []
for uid, metadata_uid in tqdm(metadata.items(), total=len(metadata)):
    fcst_df_uid = fcst_df.loc[uid]
    forecast_arrays = ufp.value_cols_to_numpy(
        df=fcst_df_uid,
        id_col="unique_id",
        time_col="ds",
        target_col=None,
    )
    forecast_keys = ["mean"]
    if pipeline.predictor.quantiles is not None:
        forecast_keys += [f"{q}" for q in pipeline.predictor.quantiles]
    q_fcst = QuantileForecast(
        forecast_arrays=forecast_arrays.T,
        forecast_keys=forecast_keys,
        item_id=metadata_uid["item_id"],
        start_date=metadata_uid["fcst_start"],
    )
    fcsts.append(q_fcst)


  0%|          | 0/42 [00:00<?, ?it/s]

In [21]:
fcsts.__len__(), fcsts[0].quantile(0.5).__len__()

(42, 48)

In [22]:
fcsts_50qtl = [fcst.quantile(0.5) for fcst in fcsts]

In [23]:
fcsts_arr = np.stack(fcsts)  # shape: (42, 48)

# Reshape to (6, 7, h)
fcsts_reshaped = fcsts_arr.reshape(6, 7, )
fcsts_reshaped.shape

(6, 7)

In [24]:
fcsts_reshaped[0][0].quantile(0.5).shape

(48,)

## Evaluate

### Evaluation Pipeline Summary

1. **Batching Data and Forecasts**  
   The original data and forecasts are passed to `gluonts.model.evaluation.evaluate_forecasts_raw`, where `data.input`, `data.label`, and forecasts are parsed and grouped into batches using a batcher.

2. **Batch Wrapping with ChainMap**  
   For each batch, the input, label, and forecasts are combined into a `ChainMap` object called `data_batch`, which is basically a merged dictionary for convenient access and is then passed to each metric (e.g., MAE, MSE) for evaluation. Here is why we need to modify the source code of `gluonts`, because in the original package `input` is only used to derive seasonality. The original `input` time series are not included in the final `data_batch`. 

3. **Metric Calculation Process**  
   - Each metric is instantiated as an evaluator, and the calculation is performed via `evaluator.update(data_batch)`.
   - The evaluator has two main components:
     - `stat`: Defines the error calculation (e.g., `absolute_error` for MAE). A `partial` wrapper is used to fix parameters such as `forecast_type='0.5'`.
     - `aggregate`: Typically a `Mean()` class with a specified axis for aggregation.
   - Under the hood, `evaluator.update` executes `aggregate.step(stat(data))`:
     - `stat(data)` computes the error for the batch (e.g., `absolute_error(data_batch, forecast_type='0.5')`).
     - `aggregate.step()` processes the error and updates:
       - `aggregate.partial_result`: The sum of all errors in the batch.
       - `aggregate.get()`: The mean error, calculated as `aggregate.partial_result / aggregate.n`.

4. **Aggregating Results Across Batches**  
   As the evaluation iterates through all batches, the evaluator accumulates the total error (`partial_result`) and the total number of time steps (`n`). The final average error across all batches is obtained via `aggregate.get()`.

### Notes
1. For multivar data, metrics are calculated with all vars flattened into one - aggregated over all time steps. Make sense when the number of steps among vars are largely different

### Preproc

In [25]:
from gluonts.ev.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    MSIS,
    ND,
    NRMSE,
    RMSE,
    SMAPE,
    MeanWeightedSumQuantileLoss,
)

from gluonts.itertools import batcher

from gluonts.model.evaluation import _get_data_batch
from toolz import first

In [26]:
test_data = evaluator.dataset.test_data
# test_data = evaluator.dataset.univar_test_data
axis = None
metrics = [
    MAE(),
]

batch_size=512 # ?

# forecasts = fcsts_reshaped
forecasts = fcsts


EVALUATOR = Evaluator(
    dataset_name=dataset_cfg.get("name"),
    term=dataset_cfg.get("term", "short"),
    to_univariate=to_univariate,
    # Create a subdirectory for each model's results
    output_path=f"{pipeline.output_path}/{pipeline.forecaster.alias}",
    storage_path=pipeline.storage_path,
    config=pipeline.config
)

seasonality = EVALUATOR.seasonality
mask_invalid_label = True
allow_nan_forecast = False

In [27]:
label_ndim = first(test_data.label)["target"].ndim

assert label_ndim in [1, 2]

if axis is None:
    axis = tuple(range(label_ndim + 1))
if isinstance(axis, int):
    axis = (axis,)

assert all(ax in range(3) for ax in axis)

evaluators = {}
evaluator_metric = MAE()(axis=axis)
evaluators[evaluator_metric.name] = evaluator_metric


index_data = []

# TODO: Here the returned batch size is actually just one, within which there are several windows. 
# We may want to align the batcher and the windows, or remove batcher altogether.
input_batches = batcher(test_data.input, batch_size=batch_size)
label_batches = batcher(test_data.label, batch_size=batch_size)
forecast_batches = batcher(forecasts, batch_size=batch_size)

# pbar = tqdm()

input_batch, label_batch, forecast_batch = first(zip(input_batches, label_batches, forecast_batches))

if 0 not in axis:
    index_data.extend(
        [
            (forecast.item_id, forecast.start_date)
            for forecast in forecast_batch
        ]
    )


data_batch = _get_data_batch(
            input_batch,
            label_batch,
            forecast_batch,
            seasonality=seasonality,
            mask_invalid_label=mask_invalid_label,
            allow_nan_forecast=allow_nan_forecast,
        )

In [28]:
input_batch[0]['target'].shape

(2376,)

In [29]:
label_batch[0]['target']

array([ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.07666667,  0.74      ,  2.9758334 ,  3.04      ,  3.04      ,
        3.04      ,  3.04      ,  3.0016668 ,  2.58      ,  3.5       ,
        5.34      ,  5.34      ,  5.34      ,  5.34      ,  5.3075    ,
        4.95      ,  4.95      ,  4.7083335 ,  5.0641665 ,  5.6433334 ,
        5.67      ,  6.23      ,  6.62      ,  7.25      , 10.098333  ,
       10.315     , 11.218333  , 11.948334  , 12.        , 13.494166  ,
       14.37      , 15.315     , 15.64      , 16.066668  , 16.22      ,
       17.3925    , 17.485834  , 17.91      , 18.26      , 18.251667  ,
       18.581667  , 19.09      , 19.284166  ], dtype=float32)

In [30]:
from gluonts.time_feature.seasonality import get_seasonality
from gluonts.ev.ts_stats import seasonal_error

label_target = np.stack([label["target"] for label in label_batch], axis=0)
if mask_invalid_label:
    label_target = np.ma.masked_invalid(label_target)

# input_target_stack = np.stack([input_["target"] for input_ in input_batch], axis=0)
input_target = [input_["target"] for input_ in input_batch]
# if mask_invalid_label:
#     input_target = np.ma.masked_invalid(input_target)

other_data = {
    "label": label_target,
    "input": input_target,
}

seasonal_error_values = []
for input_ in input_batch:
    seasonality_entry = seasonality
    if seasonality_entry is None:
        seasonality_entry = get_seasonality(input_["start"].freqstr)
    input_target = input_["target"]
    if mask_invalid_label:
        input_target = np.ma.masked_invalid(input_target)
    seasonal_error_values.append(
        seasonal_error(
            input_target,
            seasonality=seasonality_entry,
            time_axis=-1,
        )
    )
other_data["seasonal_error"] = np.array(seasonal_error_values)


In [31]:
from gluonts.model.evaluation import BatchForecast

data_batch_forecast = BatchForecast(forecast_batch, False)

In [32]:
# ChainMap is a class from Python’s collections module 
# that groups multiple dictionaries (or mappings) together to be treated as a single, updateable view. 
# When you search for a key, it looks through each dictionary in order until it finds the key.
from collections import ChainMap

dict1 = {'a': 1, 'b': 2}
dict2 = {'b': 3, 'c': 4}

cm = ChainMap(dict1, dict2)
print(cm['b'])  # Output: 2 (from dict1)
print(cm['c'])  # Output: 4 (from dict2)

2
4


### Metric Update

In [33]:
data_batch['label'].data.shape, data_batch['input'].__len__(), data_batch['input'][0].shape

((42, 48), 42, (2376,))

In [34]:
# Calculate the mean of each array in 'input'
input_means = np.array([arr.mean() for arr in other_data['input']])

# Subtract the mean from each corresponding array in label.data
label_minus_input_mean = np.array([label_arr - mean for label_arr, mean in zip(data_batch['label'], input_means)])

# label_minus_input_mean will be a (42, N) array, where N is the length of each label array
label_minus_input_mean.shape, label_minus_input_mean

((42, 48),
 array([[-16.126362 , -16.126362 , -16.126362 , ...,   2.455305 ,
           2.9636383,   3.1578045],
        [  3.6130495,   3.6855497,   4.1738825, ...,   8.364715 ,
           8.3388815,   8.455548 ],
        [  8.567972 ,   8.642971 ,   8.739637 , ...,   9.094639 ,
           9.094639 ,   9.094639 ],
        ...,
        [-24.538095 , -24.538095 , -24.538095 , ...,   4.0452385,
           5.6285706,   9.128572 ],
        [ 12.271969 ,   9.521969 ,   9.938637 , ...,  20.771969 ,
          20.1053   ,  20.1053   ],
        [ 20.603819 ,  23.770483 ,  21.020483 , ...,  27.103819 ,
          19.770483 ,  15.437151 ]], dtype=float32))

In [35]:
np.abs(data_batch['label'] - input_means[:, None])

masked_array(
  data=[[16.126361846923828, 16.126361846923828, 16.126361846923828, ...,
         2.4553050994873047, 2.9636383056640625, 3.157804489135742],
        [3.6130495071411133, 3.685549736022949, 4.173882484436035, ...,
         8.364714622497559, 8.338881492614746, 8.455548286437988],
        [8.567972183227539, 8.64297103881836, 8.73963737487793, ...,
         9.09463882446289, 9.09463882446289, 9.09463882446289],
        ...,
        [24.538095474243164, 24.538095474243164, 24.538095474243164, ...,
         4.045238494873047, 5.628570556640625, 9.128572463989258],
        [12.271968841552734, 9.521968841552734, 9.938636779785156, ...,
         20.771968841552734, 20.105300903320312, 20.105300903320312],
        [20.603818893432617, 23.770483016967773, 21.020483016967773, ...,
         27.103818893432617, 19.770483016967773, 15.437150955200195]],
  mask=[[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, 

In [115]:
# Breaking this down:
MAE()().update(data_batch)

DirectMetric(name='MAE[0.5]', stat=functools.partial(<function absolute_error at 0x700ff1d5f420>, forecast_type='0.5'), aggregate=Mean(axis=None, partial_result=array(14139.86379857), n=2016))

In [67]:
from gluonts.ev.stats import absolute_error

from gluonts.ev.metrics import DirectMetric
from gluonts.ev.aggregations import Mean
from functools import partial


In [126]:
# partial allows you to fix a certain number of arguments of a function and generate a new function.
# Here, we are fixing the forecast_type argument of absolute_error to '0.5'
stat = partial(absolute_error, forecast_type='0.5')
aggregate = Mean(axis=axis)

In [127]:
mae = DirectMetric(
            name=f"MAE['0.5']",
            stat=stat,
            aggregate=aggregate,
        )

In [128]:
error_batch = stat(data_batch)
# equivalent to: error_batch = np.abs(data_batch['label'] - data_batch['0.5'])

In [129]:
stat(data_batch) == absolute_error(data_batch, forecast_type='0.5')

masked_array(
  data=[[ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        ...,
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True]],
  mask=False,
  fill_value=True)

In [130]:
aggregate.step(error_batch)

In [131]:
print(aggregate.partial_result == sum(sum(error_batch.data)))
print(aggregate.get() == aggregate.partial_result / aggregate.n)

True
True


In [132]:
aggregate.n

array(2016.)

In [133]:
2016/42

48.0

In [124]:
evaluator_metric.update(data_batch)

DirectMetric(name='MAE[0.5]', stat=functools.partial(<function absolute_error at 0x7651374e6d40>, forecast_type='0.5'), aggregate=Mean(axis=(0, 1), partial_result=array(71368.07279547), n=array(14112.)))

In [125]:
evaluator_metric.aggregate.partial_result, evaluator_metric.aggregate.n, evaluator_metric.aggregate.get()

(array(71368.07279547), array(14112.), 5.0572613942370355)

In [114]:
print(evaluator_metric.get() == aggregate.get())

True


### Custom Metrics

In [57]:
MAR(context_length=512)(axis=axis).update(data_batch).get()

TypeError: MAR.__init__() got an unexpected keyword argument 'context_length'

In [48]:
MAE()

MAE(forecast_type='0.5')

In [49]:
MAE()()

DirectMetric(name='MAE[0.5]', stat=functools.partial(<function absolute_error at 0x716c61b3b240>, forecast_type='0.5'), aggregate=Mean(axis=None, partial_result=None, n=None))

In [55]:
MSE(forecast_type='0.5')

MSE(forecast_type='0.5')

In [60]:
MBR(forecast_type='0.5')

MBR(forecast_type='0.5')